# Notebook 04 — Model Evaluation

Full evaluation of both models on held-out test data.

**What we evaluate:**
1. BiLSTM: Accuracy, AUC-ROC, confusion matrix, calibration
2. Form scorer: MAE, R², feature importances, SHAP
3. Feedback engine: sample predictions with coaching cues
4. Error analysis: what makes the model fail?

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from pathlib import Path
from sklearn.metrics import (
    accuracy_score, roc_auc_score, roc_curve,
    confusion_matrix, classification_report,
)
from sklearn.calibration import calibration_curve

from src.modeling.bilstm_model import ShotPredictor
from src.modeling.form_scorer import FeedbackEngine, FORM_FEATURES, create_form_score_labels

sns.set_theme(style='whitegrid')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Libraries loaded | Device: {device}')

## 1. Load Pre-trained Models & Test Data

In [ ]:
seq_dir   = Path('../data/processed/sequences')
ckpt_path = Path('../models/bilstm/best_model.pt')
form_path = Path('../models/form_scorer.pkl')
rel_path  = Path('../data/processed/features/release_features.csv')

# Load test sequences
X_test = np.load(seq_dir / 'X_test.npy')
y_test = np.load(seq_dir / 'y_test.npy')
print(f'Test set: {len(X_test)} shots | Makes: {int(y_test.sum())} | Misses: {int((1-y_test).sum())}')

# Load BiLSTM
if ckpt_path.exists():
    ckpt = torch.load(ckpt_path, map_location=device)
    model = ShotPredictor(
        input_size=ckpt['input_size'],
        hidden_size=ckpt['hidden_size'],
        num_layers=ckpt['num_layers'],
        dropout=ckpt['dropout'],
    ).to(device)
    model.load_state_dict(ckpt['model_state'])
    model.eval()
    print(f'BiLSTM loaded (val AUC at save: {ckpt.get("val_auc", "N/A")})')
else:
    model = None
    print('BiLSTM checkpoint not found — run train_bilstm.py first')

# Load form scorer
if form_path.exists():
    with open(form_path, 'rb') as f:
        form_artifact = pickle.load(f)
    form_pipeline = form_artifact['pipeline']
    feedback_engine = form_artifact['feedback_engine']
    print('Form scorer loaded')
else:
    form_pipeline = None
    feedback_engine = FeedbackEngine()  # use rule-based fallback
    print('Form scorer not found — using rule-based fallback')

## 2. BiLSTM Evaluation

In [ ]:
if model is not None:
    X_t = torch.FloatTensor(X_test).to(device)
    with torch.no_grad():
        probs = model(X_t).cpu().squeeze().numpy()
    preds = (probs > 0.5).astype(int)
    
    acc = accuracy_score(y_test, preds)
    auc = roc_auc_score(y_test, probs)
    
    print('BILSTM TEST RESULTS')
    print('='*50)
    print(f'Accuracy : {acc:.4f}')
    print(f'AUC-ROC  : {auc:.4f}')
    print()
    print(classification_report(y_test, preds, target_names=['Miss', 'Make']))
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # Confusion matrix
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
               xticklabels=['Miss','Make'], yticklabels=['Miss','Make'], ax=axes[0])
    axes[0].set_title('Confusion Matrix'); axes[0].set_ylabel('True'); axes[0].set_xlabel('Pred')
    
    # ROC curve
    fpr, tpr, _ = roc_curve(y_test, probs)
    axes[1].plot(fpr, tpr, color='#2ecc71', lw=2, label=f'BiLSTM (AUC={auc:.3f})')
    axes[1].plot([0,1],[0,1], 'k--', lw=1)
    axes[1].set_title('ROC Curve'); axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
    axes[1].legend()
    
    # Calibration
    frac_pos, mean_pred = calibration_curve(y_test, probs, n_bins=8)
    axes[2].plot(mean_pred, frac_pos, 's-', color='#4C72B0', label='BiLSTM')
    axes[2].plot([0,1],[0,1], 'k--', label='Perfect calibration')
    axes[2].set_title('Calibration Curve'); axes[2].legend()
    
    plt.suptitle('BiLSTM — Test Set Evaluation', fontsize=13, fontweight='bold')
    plt.tight_layout(); plt.show()

## 3. Form Scorer + Feedback Demo

In [ ]:
if rel_path.exists():
    release_df = pd.read_csv(rel_path)
    
    # Demo: pick a few shots and show feedback
    print('FEEDBACK ENGINE DEMO')
    print('='*70)
    
    for _, row in release_df.sample(min(3, len(release_df)), random_state=42).iterrows():
        feats = row.to_dict()
        feedback = feedback_engine.generate(feats)
        rule_score = feedback_engine.score_form(feats)
        
        outcome_label = 'MAKE ✅' if row['outcome'] == 1 else 'MISS ❌'
        print(f'\nShot: {row["video_stem"][:40]} | Actual: {outcome_label}')
        print(f'Form Score (rule-based): {rule_score:.1f}/100')
        if feedback:
            for item in feedback:
                icon = '🔴' if item['severity'] == 'high' else '🟡'
                print(f'  {icon} [{item["severity"].upper()}] {item["message"]}')
        else:
            print('  ✅ No major form issues detected!')
        print('-'*70)

## 4. SHAP Values — Form Scorer Explainability

In [ ]:
if form_pipeline is not None and rel_path.exists():
    try:
        import shap
        release_df = pd.read_csv(rel_path)
        avail = [f for f in FORM_FEATURES if f in release_df.columns]
        X = release_df[avail].fillna(0)
        
        clf = form_pipeline.named_steps['reg']
        scaler = form_pipeline.named_steps['scaler']
        X_scaled = pd.DataFrame(scaler.transform(X), columns=avail)
        
        explainer = shap.TreeExplainer(clf)
        shap_vals = explainer.shap_values(X_scaled)
        
        plt.figure(figsize=(10, 6))
        shap.summary_plot(shap_vals, X_scaled, show=False)
        plt.title('SHAP Summary — Form Scorer Feature Importance', fontsize=13, fontweight='bold')
        plt.tight_layout(); plt.show()
        
        print('Top features driving form score (by mean |SHAP|):')
        mean_shap = np.abs(shap_vals).mean(axis=0)
        for feat, val in sorted(zip(avail, mean_shap), key=lambda x: -x[1])[:8]:
            print(f'  {feat:40s}: {val:.4f}')
    except ImportError:
        print('shap not installed — pip install shap')
else:
    print('Load form scorer and release features first.')

## 5. Results Summary

In [ ]:
eval_path = Path('../docs/evaluation/evaluation_results.json')
if eval_path.exists():
    with open(eval_path) as f:
        results = json.load(f)
else:
    results = {}

print('FINAL EVALUATION SUMMARY')
print('='*50)
summary = pd.DataFrame([
    {'Component': 'BiLSTM', 'Metric': 'Accuracy',   'Value': results.get('bilstm_accuracy', 'run evaluate.py')},
    {'Component': 'BiLSTM', 'Metric': 'AUC-ROC',    'Value': results.get('bilstm_auc',      'run evaluate.py')},
    {'Component': 'Form Scorer', 'Metric': 'MAE',   'Value': results.get('form_mae',         'run evaluate.py')},
    {'Component': 'Form Scorer', 'Metric': 'R²',    'Value': results.get('form_r2',          'run evaluate.py')},
])
print(summary.to_string(index=False))